In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from config import PATHS, TABLES, CHECKPOINTS
from metrics_utils import log_pipeline_metric

In [0]:
from pyspark.sql import functions as F

def log_pipeline_metric(spark, pipeline_name, df):

    metrics_df = spark.createDataFrame(
        [(pipeline_name, df.count())],
        ["pipeline", "row_count"]
    ).withColumn("processed_at", F.current_timestamp())

    metrics_df.write.mode("append").saveAsTable(
        "dbw_retails.monitoring.pipeline_metrics"
    )

In [0]:
schema = StructType([
    StructField("InvoiceNo", StringType()),
    StructField("StockCode", StringType()),
    StructField("Description", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("InvoiceDate", StringType()),
    StructField("UnitPrice", DoubleType()),
    StructField("CustomerID", DoubleType()),
    StructField("Country", StringType())
])

df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.useNotifications","false")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", PATHS["bronze_files"] + "_schema")
    .option("cloudFiles.maxFilesPerTrigger", 100)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("rescuedDataColumn", "_rescued_data")
    .schema(schema)
    .load(PATHS["raw"])
)

In [0]:
df_stream = df_stream.drop("index")

In [0]:
from pyspark.sql.functions import col

df_stream = df_stream.select(
    col("InvoiceNo"),
    col("StockCode"),
    col("Description"),
    col("Quantity").cast("int"),
    col("InvoiceDate"),
    col("UnitPrice").cast("double"),
    col("CustomerID").cast("double"),
    col("Country")
)

In [0]:
from pyspark.sql.functions import current_timestamp, col

df_stream = (
    df_stream
    .withColumn("_ingest_ts", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

In [0]:
from pyspark.sql.functions import current_timestamp, col

df_stream = (
    df_stream
    .withColumn("_ingest_ts", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

In [0]:
def process_batch(batch_df, batch_id):

    batch_df.write \
        .format("delta") \
        .mode("append") \
        .option("txnAppId", "bronze_files") \
        .option("txnVersion", batch_id) \
        .save(PATHS["bronze_files"])

    log_pipeline_metric(spark, "bronze_files", batch_df)

In [0]:
query = (
    df_stream.writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", CHECKPOINTS["bronze_files"])
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
spark.read.format("csv").option("header",True).load(PATHS["raw"])

DataFrame[index: string, InvoiceNo: string, StockCode: string, Description: string, Quantity: string, InvoiceDate: string, UnitPrice: string, CustomerID: string, Country: string]